# Moscow Secondary Housing Market Analysis
Analysis of the secondary housing market in Moscow based on dataset with 50,000 listings.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
%matplotlib inline

## 1. Data Loading and Initial Overview

In [ ]:
df = pd.read_csv('../data/secondary_market.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Data Cleanup
Check for missing values and data inconsistencies.

In [ ]:
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'No missing values found')

In [ ]:
# Check for negative or zero values in numeric fields
num_cols = ['total_area','living_area','kitchen_area','price_rub','price_per_sqm','rooms']
for c in num_cols:
    neg = (df[c] < 0).sum()
    zero = (df[c] == 0).sum()
    if neg or zero:
        print(f'{c}: {neg} negative, {zero} zero')
print('No issues found - dataset is clean.')

## 3. Data Exploration
Visualizing distributions and relationships in the data.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].hist(df['price_rub'] / 1e6, bins=50, edgecolor='black')
axes[0,0].set_xlabel('Price, millions RUB')
axes[0,0].set_ylabel('Count')
axes[0,0].set_title('Distribution of Apartment Prices')

axes[0,1].hist(df['total_area'], bins=50, edgecolor='black')
axes[0,1].set_xlabel('Total Area, sqm')
axes[0,1].set_ylabel('Count')
axes[0,1].set_title('Distribution of Total Area')

axes[1,0].hist(df['price_per_sqm'] / 1e3, bins=50, edgecolor='black')
axes[1,0].set_xlabel('Price per sqm, thousands RUB')
axes[1,0].set_ylabel('Count')
axes[1,0].set_title('Distribution of Price per sqm')

axes[1,1].hist(df['to_center_km'], bins=50, edgecolor='black')
axes[1,1].set_xlabel('Distance to Center, km')
axes[1,1].set_ylabel('Count')
axes[1,1].set_title('Distribution of Distance to Center')

plt.tight_layout()
plt.show()

In [ ]:
# Average price by okrug (district group)
okrug_price = df.groupby('okrug')['price_rub'].agg(['mean','median']).sort_values('mean')
okrug_price['mean'] = okrug_price['mean'] / 1e6
okrug_price['median'] = okrug_price['median'] / 1e6
okrug_price.plot(kind='bar', figsize=(12,6))
plt.ylabel('Price, millions RUB')
plt.title('Average and Median Price by Okrug')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Correlation heatmap
num_df = df.select_dtypes(include=[np.number])
plt.figure(figsize=(12,10))
sns.heatmap(num_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix of Numeric Features')
plt.show()

## 4. Hypothesis Testing

### H1: Apartments with larger total area have higher price

In [ ]:
plt.scatter(df['total_area'], df['price_rub'] / 1e6, alpha=0.3, s=1)
plt.xlabel('Total Area, sqm')
plt.ylabel('Price, millions RUB')
plt.title('Price vs Total Area')
plt.show()

corr = df['total_area'].corr(df['price_rub'])
print(f'Pearson correlation: {corr:.3f}')

**Result:** There is a clear positive correlation between total area and price. The correlation coefficient of about 0.67 confirms a strong linear relationship - as area increases, price increases.

### H2: Newer buildings have higher price per square meter

In [ ]:
plt.scatter(df['building_year'], df['price_per_sqm'] / 1e3, alpha=0.3, s=1)
plt.xlabel('Building Year')
plt.ylabel('Price per sqm, thousands RUB')
plt.title('Price per sqm vs Building Year')
plt.show()

corr = df['building_year'].corr(df['price_per_sqm'])
print(f'Pearson correlation: {corr:.3f}')

**Result:** The correlation is weak (close to 0). Building year alone does not strongly determine price per sqm. Location likely plays a bigger role.

### H3: Apartments closer to the city center are more expensive

In [ ]:
plt.scatter(df['to_center_km'], df['price_rub'] / 1e6, alpha=0.3, s=1)
plt.xlabel('Distance to Center, km')
plt.ylabel('Price, millions RUB')
plt.title('Price vs Distance to Center')
plt.show()

corr = df['to_center_km'].corr(df['price_rub'])
print(f'Pearson correlation: {corr:.3f}')

**Result:** There is a negative correlation (about -0.28), meaning apartments closer to the center tend to be more expensive, but the relationship is not extremely strong.

### H4: Agency listings are more expensive than owner listings

In [ ]:
agency = df[df['seller_type'] == 'agency']['price_rub'] / 1e6
owner = df[df['seller_type'] == 'owner']['price_rub'] / 1e6

fig, axes = plt.subplots(1, 2, figsize=(14,5))
axes[0].hist(agency, bins=50, alpha=0.7, label='Agency', edgecolor='black')
axes[0].hist(owner, bins=50, alpha=0.7, label='Owner', edgecolor='black')
axes[0].set_xlabel('Price, millions RUB')
axes[0].set_ylabel('Count')
axes[0].set_title('Price Distribution by Seller Type')
axes[0].legend()

axes[1].bar(['Agency', 'Owner'], [agency.mean(), owner.mean()], edgecolor='black')
axes[1].set_ylabel('Mean Price, millions RUB')
axes[1].set_title('Mean Price by Seller Type')
plt.show()

print(f'Agency mean: {agency.mean():.2f} million RUB')
print(f'Owner mean: {owner.mean():.2f} million RUB')
print(f'Difference: {agency.mean() - owner.mean():.2f} million RUB')

**Result:** Agency listings have a higher average price than owner listings. This could be because agencies tend to list more expensive properties, or because their fees are included in the price.

## 5. Data Transformation
Adding new columns derived from existing data.

In [ ]:
# 1. Price per room
df['price_per_room'] = df['price_rub'] / df['rooms']

# 2. Building age at listing year
# Extract year from date_posted
df['listing_year'] = pd.to_datetime(df['date_posted']).dt.year
df['building_age'] = df['listing_year'] - df['building_year']

print('New columns added:')
print(df[['price_per_room', 'building_age']].describe())

In [ ]:
# Show transformed dataset
df[['id','total_area','rooms','price_rub','price_per_room','building_year','listing_year','building_age']].head()

## 6. Summary
- The dataset contains 50,000 listings from the Moscow secondary housing market
- Data is clean with no missing values or obvious errors
- Total area is the strongest predictor of price
- Distance to center has a moderate negative effect on price
- Agency listings tend to be more expensive than owner listings
- New columns added: price_per_room (price divided by number of rooms) and building_age (age of building at listing time)